In [ ]:
import dataclasses

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import optika
import furst

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="function 'sqrt' is not known")

In [ ]:
axis_channel = "channel"
axis_wavelength = "wavelength"
axis_field = ("field_x", "field_y")
axis_pupil = ("pupil_x", "pupil_y")

num_field = 15
num_pupil = 15

instrument = furst.instruments.design(num_wavelength=5)

# a separate draw for each channel
ones = na.ScalarArray(
    ndarray=np.ones(instrument.feed_optic.rowland_azimuth.shape[axis_channel]),
    axes=(axis_channel,),
)

instrument.field = na.Cartesian2dVectorStratifiedRandomSpace(
    start=-ones,
    stop=+ones,
    axis=na.Cartesian2dVectorArray(*axis_field),
    num=num_field,
    centers=True,
    seed=42,
)
instrument.pupil = na.Cartesian2dVectorStratifiedRandomSpace(
    start=-ones,
    stop=+ones,
    axis=na.Cartesian2dVectorArray(*axis_pupil),
    num=num_pupil,
    centers=True,
    seed=43,
)

system = instrument.system
axis_surface = system.axis_surface

In [ ]:
azimuth = instrument.feed_optic.rowland_azimuth
wavelength_min = instrument.wavelength_min.to(u.nm)
wavelength_max = instrument.wavelength_max.to(u.nm)

for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(
        f"channel {i}:"
        f" feed optic at {azimuth[index].ndarray:.2f},"
        f" {wavelength_min[index].ndarray:.1f} to {wavelength_max[index].ndarray:.1f}"
    )

In [ ]:
instrument_sparse = furst.instruments.design(
    num_wavelength=3,
    num_field=3,
    num_pupil=3,
)

fig, axs = plt.subplots(
    nrows=2,
    sharex=True,
    figsize=(9, 6),
    constrained_layout=True,
)
for ax, components in zip(axs, [("z", "x"), ("z", "y")]):
    instrument_sparse.system.plot(
        ax=ax,
        components=components,
        color="black",
        kwargs_rays=dict(
            color="tab:blue",
            linewidth=0.5,
        ),
    )
    ax.set_ylabel(f"${components[1]}$ ({ax.get_ylabel()})")
axs[1].set_xlabel(f"$z$ ({axs[1].get_xlabel()})")
axs[0].set_title("top view")
axs[1].set_title("side view")
axs[0].set_aspect("equal")

In [ ]:
raytrace = system.raytrace()
unvignetted = raytrace.outputs.unvignetted

# looked up by name, since the filter adds surfaces ahead of the detector
name_surface = [surface.name for surface in system.surfaces_all]
index_surface = {
    "feed optic": name_surface.index("feed optic"),
    "grating": name_surface.index("grating"),
    "detector": name_surface.index("sensor"),
}

fig, axs = plt.subplots(
    ncols=len(index_surface),
    figsize=(10, 4),
    constrained_layout=True,
)
for ax, name in zip(axs, index_surface):
    index = {axis_surface: index_surface[name]}
    surface = system.surfaces_all[index_surface[name]]
    position = surface.transformation.inverse(raytrace.outputs.position[index])
    na.plt.scatter(
        position.x,
        position.y,
        ax=ax,
        where=unvignetted[index],
        s=1,
    )
    wire = surface.aperture.wire()
    na.plt.plot(
        wire.x,
        wire.y,
        ax=ax,
        axis="wire",
        color="black",
    )
    ax.set_title(name)
    if name != "feed optic":
        ax.set_aspect("equal")

In [ ]:
rays = system.rayfunction_default.outputs

weight = rays.unvignetted.astype(float)
axes_disk = axis_field + axis_pupil

width_pixel = system.sensor.width_pixel

position = rays.position
position_mean = (position * weight).sum(axes_disk) / weight.sum(axes_disk)

dx = (position.x - position_mean.x) / width_pixel
dx = dx.to(u.dimensionless_unscaled) * u.pix

In [ ]:
index_center = {axis_wavelength: instrument.wavelength.shape[axis_wavelength] // 2}

lsf_2d = na.histogram2d(
    dx[index_center],
    position.y[index_center],
    bins=dict(lsf_x=21, lsf_y=21),
    axis=axes_disk,
    weights=weight[index_center],
    min=na.Cartesian2dVectorArray(-1 * u.pix, -8 * u.mm),
    max=na.Cartesian2dVectorArray(+1 * u.pix, +8 * u.mm),
)

fig, axs = na.plt.subplots(
    axis_cols=axis_channel,
    ncols=azimuth.shape[axis_channel],
    sharex=True,
    sharey=True,
    figsize=(10, 3.5),
    constrained_layout=True,
)
na.plt.pcolormesh(
    C=lsf_2d,
    ax=axs,
)
for i, ax in enumerate(axs.ndarray):
    ax.set_title(f"{azimuth[{axis_channel: i}].ndarray:.1f}")

In [ ]:
lsf = na.histogram(
    dx,
    bins=dict(lsf_x=41),
    axis=axes_disk,
    weights=weight,
    min=-1 * u.pix,
    max=+1 * u.pix,
)

fig, axs = na.plt.subplots(
    axis_cols=axis_channel,
    ncols=azimuth.shape[axis_channel],
    sharex=True,
    sharey=True,
    figsize=(10, 3),
    constrained_layout=True,
)
na.plt.stairs(
    lsf.inputs,
    lsf.outputs,
    ax=axs,
    axis="lsf_x",
)
for i, ax in enumerate(axs.ndarray):
    ax.set_title(f"{azimuth[{axis_channel: i}].ndarray:.1f}")
axs.ndarray[0].set_ylabel("rays");

In [ ]:
wavelength = instrument.wavelength_physical.to(u.nm)

variance_geometric = (np.square(dx) * weight).sum(axes_disk) / weight.sum(axes_disk)
variance_pixel = np.square(1 * u.pix) / 12
width = np.sqrt(variance_geometric + variance_pixel)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.plot(
        wavelength[index],
        width[index],
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
ax.set_ylabel(f"LSF width ({ax.get_ylabel()})")
ax.legend(title="feed optic azimuth");

In [ ]:
index_first = {axis_wavelength: 0}
index_last = {axis_wavelength: ~0}
dispersion = wavelength[index_last] - wavelength[index_first]
dispersion = dispersion / (position_mean.x[index_last] - position_mean.x[index_first])

wavelength_resolvable = 2 * width * (width_pixel / u.pix) * dispersion
resolving_power = (wavelength / wavelength_resolvable).to(u.dimensionless_unscaled)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.plot(
        wavelength[index],
        resolving_power[index],
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
ax.set_ylabel("resolving power")
ax.legend(title="feed optic azimuth");

In [ ]:
print(f"minimum resolving power: {resolving_power.min().ndarray:.0f}")
print(f"mean resolving power: {resolving_power.mean().ndarray:.0f}")
print(f"maximum resolving power: {resolving_power.max().ndarray:.0f}")

In [ ]:
fraction = unvignetted.mean(axes_disk + (axis_wavelength,))
names = name_surface

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    ax.plot(
        names,
        fraction[index].ndarray,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_ylim(bottom=0)
ax.set_ylabel("fraction of rays surviving")
ax.legend(title="feed optic azimuth");

In [ ]:
index_grating = {axis_surface: index_surface["grating"]}
index_detector = {axis_surface: index_surface["detector"]}

position_detector = system.sensor.transformation.inverse(
    raytrace.outputs.position[index_detector]
)

half_height_detector = instrument.camera.sensor.num_pixel_active.y * width_pixel / 2
half_height_detector = half_height_detector.to(u.mm)

distribution_y = na.histogram(
    position_detector.y,
    bins=dict(lsf_y=41),
    axis=axes_disk + (axis_wavelength,),
    weights=unvignetted[index_grating].astype(float),
    min=-20 * u.mm,
    max=+20 * u.mm,
)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.stairs(
        distribution_y.inputs,
        distribution_y.outputs[index],
        ax=ax,
        axis="lsf_y",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.axvline(-half_height_detector, color="black", linestyle="--")
ax.axvline(+half_height_detector, color="black", linestyle="--")
ax.set_xlabel(f"height on the detector ({ax.get_xlabel()})")
ax.set_ylabel("rays")
ax.legend(title="feed optic azimuth");

In [ ]:
axis_position = "position"
translation = na.linspace(-1, 2, axis=axis_position, num=13) * u.mm
angle = na.linspace(-0.3, 0.3, axis=axis_position, num=13) * u.deg

# five wavelengths across each channel, in normalized coordinates so that
# every channel gets its own range
wavelength_focus = na.linspace(
    start=instrument.wavelength.min(),
    stop=instrument.wavelength.max(),
    axis=axis_wavelength,
    num=5,
)


def moved(**kwargs):
    """The instrument with its feed optic array moved."""
    return dataclasses.replace(
        instrument,
        feed_optic=dataclasses.replace(instrument.feed_optic, **kwargs),
    )


def width_channel(instrument, channel):
    """The width of the lines of one channel, combined in quadrature."""
    width = instrument.width_line(wavelength_focus)
    width = width[{axis_channel: channel}]
    return np.sqrt(np.square(width).mean(axis_wavelength))


width_translation = width_channel(
    instrument=moved(translation_focus=translation, angle_focus=0 * u.deg),
    channel=0,
)
width_angle = width_channel(
    instrument=moved(
        translation_focus=furst.instruments.translation_focus,
        angle_focus=angle,
    ),
    channel=~0,
)

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        figsize=(9, 3.5),
        constrained_layout=True,
    )
    curves = [
        (translation, width_translation, furst.instruments.translation_focus),
        (angle, width_angle, furst.instruments.angle_focus),
    ]
    for ax, (position, width, best) in zip(axs, curves):
        na.plt.plot(position, width, ax=ax, color="black", marker="o", markersize=3)
        ax.axvline(
            best.to_value(position.unit),
            color="tab:blue",
            linestyle="dashed",
        )
        ax.set_xlabel(f"position ({ax.get_xlabel()})")
    axs[0].set_ylabel(f"width of the lines ({axs[0].get_ylabel()})")
    axs[1].set_ylabel("")
    axs[0].set_title("lower stage, first channel")
    axs[1].set_title("upper stage, last channel")

print(f"the array slides  {furst.instruments.translation_focus:+.4f}")
print(f"the array pivots  {furst.instruments.angle_focus:+.5f}")

In [ ]:
print("line spread function of each channel, in quadrature (pixels)")
for i in range(azimuth.shape[axis_channel]):
    unfocused = width_channel(
        instrument=moved(translation_focus=0 * u.mm, angle_focus=0 * u.deg),
        channel=i,
    )
    focused = width_channel(instrument=instrument, channel=i)
    print(
        f"  channel {i}:"
        f" {unfocused.ndarray:6.3f} unfocused,"
        f" {focused.ndarray:5.3f} focused"
    )

In [ ]:
coating = furst.feed_optics.materials.coating_witness_measured()
measurement = coating.efficiency_measured

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        figsize=(9, 3.5),
        constrained_layout=True,
    )
    for ax, (limit_min, limit_max) in zip(axs, [(115, 605), (118, 200)]):
        na.plt.scatter(
            measurement.inputs.wavelength,
            measurement.outputs,
            ax=ax,
            s=10,
            color="black",
        )
        ax.axvspan(
            instrument.wavelength_min.min().ndarray.to_value(u.nm),
            instrument.wavelength_max.max().ndarray.to_value(u.nm),
            color="gray",
            alpha=0.12,
            zorder=0,
        )
        ax.set_xlim(limit_min, limit_max)
        ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    axs[0].set_ylabel("reflectance")
    axs[0].set_title("witness sample")
    axs[1].set_title("the FURST bandpass, shaded")

In [ ]:
# the angles at which the feed optics are actually illuminated, taken
# from the raytrace above
index_feed = {axis_surface: index_surface["feed optic"]}
surface_feed = system.surfaces_all[index_surface["feed optic"]]

position_feed = surface_feed.transformation.inverse(
    raytrace.outputs.position[index_feed]
)
direction_feed = surface_feed.transformation.transformation_linear.inverse(
    raytrace.outputs.direction[index_feed]
)
cosine = np.abs(direction_feed @ surface_feed.sag.normal(position_feed))
angle_feed = np.arccos(np.clip(na.value(cosine), -1, 1)) * u.rad
angle_feed = angle_feed[unvignetted[index_feed]].to(u.deg)

print(
    f"feed optic angle of incidence: {angle_feed.min().ndarray:.1f}"
    f" to {angle_feed.max().ndarray:.1f}, mean {angle_feed.mean().ndarray:.1f}"
)

# the reflectance each channel sees, the measurement interpolated to
# the traced wavelengths
reflectance = coating.efficiency(
    rays=optika.rays.RayVectorArray(
        wavelength=instrument.wavelength_physical,
        direction=na.Cartesian3dVectorArray(0, 0, 1),
    ),
    normal=na.Cartesian3dVectorArray(0, 0, -1),
)
reflectance_channel = reflectance.mean(axis_wavelength)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(f"channel {i}: mean reflectance {reflectance_channel[index].ndarray:.3f}")

In [ ]:
depth_grooves = 42 * u.nm

coating_grating = furst.gratings.materials.coating_measured()
coating_assumed = furst.gratings.materials.coating_simulated()
rulings = furst.gratings.rulings.rulings_simulated(depth_grooves)

# optika's own model of a sinusoidal profile of the same depth
rulings_optika = optika.rulings.SinusoidalRulings(
    spacing=rulings.spacing,
    depth=depth_grooves,
    diffraction_order=rulings.diffraction_order,
)

measurement_grating = coating_grating.efficiency_measured
wavelength_grating = measurement_grating.inputs.wavelength

angle_simulated = furst.gratings.rulings.angle_simulated
rays_grating = optika.rays.RayVectorArray(
    wavelength=wavelength_grating,
    direction=na.Cartesian3dVectorArray(
        x=np.sin(angle_simulated),
        y=0,
        z=np.cos(angle_simulated),
    ),
)
normal = na.Cartesian3dVectorArray(0, 0, -1)
reflectance_assumed = coating_assumed.efficiency(rays_grating, normal)

# the model of the grating, on the wavelengths Zeiss simulated
wavelength_model = rulings.efficiency_measured.inputs.wavelength
rays_model = optika.rays.RayVectorArray(
    wavelength=wavelength_model,
    position=na.Cartesian3dVectorArray(0, 0, 0) * u.mm,
    direction=rays_grating.direction,
)
efficiency_grooves = rulings.efficiency(rays_model, normal)
efficiency_grooves_optika = rulings_optika.efficiency(rays_model, normal)
efficiency_grating = efficiency_grooves * coating_grating.efficiency(rays_model, normal)

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        figsize=(13, 3.5),
        constrained_layout=True,
    )
    na.plt.plot(
        wavelength_grating,
        measurement_grating.outputs,
        ax=axs[0],
        axis="wavelength",
        color="black",
        label="test piece, measured",
    )
    na.plt.plot(
        wavelength_grating,
        reflectance_assumed,
        ax=axs[0],
        axis="wavelength",
        color="tab:orange",
        label="assumed in the simulation",
    )
    for depth in furst.gratings.rulings.depths_simulated:
        efficiency = furst.gratings.rulings.efficiency_simulated(depth)
        na.plt.plot(
            efficiency.inputs.wavelength,
            efficiency.outputs,
            ax=axs[1],
            axis="wavelength",
            linewidth=1,
            label=f"Zeiss simulation, {depth:latex_inline}",
        )
    na.plt.plot(
        wavelength_model,
        efficiency_grating,
        ax=axs[1],
        axis="wavelength",
        color="black",
        linewidth=2,
        label="model, grooves times measured coating",
    )
    na.plt.plot(
        wavelength_model,
        efficiency_grooves,
        ax=axs[2],
        axis="wavelength",
        color="black",
        label="Zeiss simulation, coating divided out",
    )
    na.plt.plot(
        wavelength_model,
        efficiency_grooves_optika,
        ax=axs[2],
        axis="wavelength",
        color="tab:red",
        linestyle="dashed",
        label="optika, thin sinusoidal grating",
    )
    for ax in axs:
        ax.axvspan(
            instrument.wavelength_min.min().ndarray.to_value(u.nm),
            instrument.wavelength_max.max().ndarray.to_value(u.nm),
            color="gray",
            alpha=0.12,
            zorder=0,
        )
        ax.set_xlim(118, 200)
        ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
        ax.legend(fontsize=8)
    axs[0].set_ylabel("reflectance")
    axs[0].set_title("grating coating")
    axs[1].set_ylabel("efficiency")
    axs[1].set_title("grating, first order")
    axs[2].set_ylabel("efficiency")
    axs[2].set_title("grooves alone, first order")

In [ ]:
rays = optika.rays.RayVectorArray(
    wavelength=instrument.wavelength_physical,
    direction=na.Cartesian3dVectorArray(0, 0, 1),
)
throughput_feed = coating.efficiency(rays, normal)
throughput_grating = coating_grating.efficiency(rays, normal) * rulings.efficiency(
    rays, normal
)
throughput = (throughput_feed * throughput_grating).mean(axis_wavelength)

print("mean throughput of the feed optic and the grating together")
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(f"  channel {i}: {throughput[index].ndarray:.3f}")

In [ ]:
blind_filter = instrument.filter
filter_material = blind_filter.material
measurement_filter = filter_material.efficiency_measured

wavelength_filter = na.linspace(118, 200, axis=axis_wavelength, num=201) * u.nm
shift_filter = blind_filter.focus_shift(wavelength_filter)

with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        figsize=(9, 3.5),
        constrained_layout=True,
    )
    na.plt.plot(
        measurement_filter.inputs.wavelength,
        measurement_filter.outputs,
        ax=axs[0],
        color="black",
    )
    na.plt.plot(
        wavelength_filter,
        shift_filter,
        ax=axs[1],
        color="black",
    )
    for ax in axs:
        ax.axvspan(
            instrument.wavelength_min.min().ndarray.to_value(u.nm),
            instrument.wavelength_max.max().ndarray.to_value(u.nm),
            color="gray",
            alpha=0.12,
            zorder=0,
        )
        ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    axs[0].set_ylabel("transmission")
    axs[0].set_title("witness sample")
    axs[1].set_ylabel(f"focus shift ({axs[1].get_ylabel()})")
    axs[1].set_title(f"{blind_filter.thickness.to(u.mm):.2f} of magnesium fluoride")

In [ ]:
throughput_filter = filter_material.efficiency(rays, normal)
throughput_total = (throughput_feed * throughput_grating * throughput_filter).mean(
    axis_wavelength
)
throughput_filter = throughput_filter.mean(axis_wavelength)

print("mean transmission of the filter, and throughput of all three together")
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(
        f"  channel {i}:"
        f" filter {throughput_filter[index].ndarray:.3f},"
        f" total {throughput_total[index].ndarray:.3f}"
    )

In [ ]:
num_field_area = 6
num_pupil_area = 11

instrument_area = dataclasses.replace(
    instrument,
    wavelength=na.linspace(
        start=instrument.wavelength.min(),
        stop=instrument.wavelength.max(),
        axis=axis_wavelength,
        num=21,
    ),
)

model = instrument_area.system.area_effective(
    field=na.Cartesian2dVectorLinearSpace(
        start=-1,
        stop=1,
        axis=na.Cartesian2dVectorArray(*axis_field),
        num=num_field_area,
    ),
    pupil=na.Cartesian2dVectorLinearSpace(
        start=-1,
        stop=1,
        axis=na.Cartesian2dVectorArray(*axis_pupil),
        num=num_pupil_area,
    ),
    seed=0,
)

wavelength_area = model.wavelength.to(u.nm)
area = model.area.to(u.mm ** 2)

with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    na.plt.plot(
        wavelength_area,
        area,
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        markersize=3,
        label=np.round(azimuth.to(u.deg), 1),
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax.set_ylabel(f"effective area ({ax.get_ylabel()})")
    ax.legend(title="feed optic azimuth", fontsize="small", ncols=2)

In [ ]:
rays_area = optika.rays.RayVectorArray(
    wavelength=wavelength_area,
    direction=na.Cartesian3dVectorArray(0, 0, 1),
)

efficiency_area = coating.efficiency(rays_area, normal)
efficiency_area = efficiency_area * coating_grating.efficiency(rays_area, normal)
efficiency_area = efficiency_area * rulings.efficiency(rays_area, normal)
efficiency_area = efficiency_area * filter_material.efficiency(rays_area, normal)
absorbance = system.sensor.material.efficiency(rays_area, normal)

area_geometric = area / (efficiency_area * absorbance)

print("channel   optics   sensor   effective area   collecting area")
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(
        f"  {i}       {efficiency_area[index].mean(axis_wavelength).ndarray:.4f}"
        f"   {absorbance[index].mean(axis_wavelength).ndarray:.3f}"
        f"     {area[index].mean(axis_wavelength).ndarray:10.4f}"
        f"     {area_geometric[index].mean(axis_wavelength).ndarray:8.2f}"
    )

In [ ]:
index_detector = {axis_surface: index_surface["detector"]}
collected = raytrace.outputs.intensity[index_detector]
unvignetted_sensor = unvignetted[index_detector]

illuminated = unvignetted_sensor.any(axis_pupil)
collected = collected.sum(axis_pupil, where=unvignetted_sensor)

# the field samples which lie within the solar disk
field_sample = instrument.field.explicit
inside = np.square(field_sample.x) + np.square(field_sample.y) <= 1

print("part of the solar disk which reaches the detector, and how evenly")
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i, axis_wavelength: 0}
    disk = inside[{axis_channel: i}]
    lit = illuminated[index] & disk
    c = collected[index]
    mean = c.mean(where=lit)
    deviation = np.sqrt(np.square(c - mean).mean(where=lit)) / mean
    print(
        f"  channel {i}:"
        f" {(lit.sum() / disk.sum()).ndarray:6.1%} of the disk,"
        f" varying by {deviation.ndarray:.1%} across it"
    )